# 02 — Instruction-Tuning Data Shape & LoRA Fine-Tuning Demo (from scratch, offline)

Companion notebook to `../02-instruction-tuning-and-llm-finetuning.md`.

Two things, both fully offline:

1. A tiny **synthetic instruction-tuning dataset** in the exact `{instruction, input, output}`
   triple shape used to specialize a model like LLaMA 3 for "summarize this protocol section in
   plain language" — the PLPS use case from this project.
2. A **from-scratch numpy demonstration of LoRA** (Low-Rank Adaptation): what `W + B·A` actually
   computes, why it needs far fewer trainable parameters than a full weight update, and a toy
   training loop showing a low-rank adapter can closely recover a low-rank target update.

No GPU, no API keys, no model downloads. Any real `transformers`/`peft` usage is wrapped in a
guarded `try/except` — this notebook explains and demonstrates the *concept*, it does not fine-
tune a real LLaMA 3 checkpoint.

## 1. Instruction dataset: the `{instruction, input, output}` shape

Each example below represents one desired behavior of an instruction-tuned PLPS-generation
model: given a fixed **instruction** (task + constraints) and a variable **input** (a protocol
section), produce a plain-language **output**. A real dataset for this task would have
thousands of examples, drawn from historical human-written PLPS documents plus synthetic
augmentation (see Chapter 02) — here we hand-write three to make the shape concrete.

In [ ]:
INSTRUCTION_DATASET = [
    {
        "instruction": (
            "Summarize the following clinical trial protocol section in plain language for a "
            "patient reading a Plain Language Protocol Synopsis (PLPS). Preserve all eligibility "
            "conditions. Write at an 8th-grade reading level."
        ),
        "input": (
            "3.1 Inclusion Criteria\n"
            "- Adults aged 18 to 75 years, inclusive, at the time of screening.\n"
            "- Confirmed diagnosis of moderate to severe persistent asthma for at least 12 months."
        ),
        "output": (
            "To join this study, you must be between 18 and 75 years old, and you must have been "
            "diagnosed with moderate to severe asthma that has lasted at least a year."
        ),
    },
    {
        "instruction": (
            "Summarize the following clinical trial protocol section in plain language for a "
            "patient reading a Plain Language Protocol Synopsis (PLPS). Preserve all eligibility "
            "conditions. Write at an 8th-grade reading level."
        ),
        "input": (
            "3.2 Exclusion Criteria\n"
            "- Current smoker or history of smoking exceeding 10 pack-years.\n"
            "- Known hypersensitivity to the study drug or any of its excipients."
        ),
        "output": (
            "You cannot join this study if you currently smoke or have a long history of smoking, "
            "or if you have had an allergic reaction to this drug or its ingredients before."
        ),
    },
    {
        "instruction": (
            "Summarize the following clinical trial protocol section in plain language for a "
            "patient reading a Plain Language Protocol Synopsis (PLPS). Preserve all eligibility "
            "conditions. Write at an 8th-grade reading level."
        ),
        "input": (
            "5.1 Dose and Regimen\n"
            "Participants weighing less than 60 kg will receive 100 mg subcutaneously every 4 "
            "weeks. Participants weighing 60 kg or more will receive 200 mg subcutaneously every "
            "4 weeks."
        ),
        "output": (
            "You will get a shot under your skin every 4 weeks. The amount of medicine depends on "
            "your body weight: a smaller dose if you weigh less than 60 kg, a larger dose if you "
            "weigh 60 kg or more."
        ),
    },
]

for i, ex in enumerate(INSTRUCTION_DATASET):
    print(f"example {i}: instruction={len(ex['instruction'])} chars, "
          f"input={len(ex['input'])} chars, output={len(ex['output'])} chars")

assert all(set(ex.keys()) == {"instruction", "input", "output"} for ex in INSTRUCTION_DATASET)
print("\nOK: every example has exactly the (instruction, input, output) triple shape.")

Notice every example shares the *same instruction* (the task definition) but a
*different input/output pair* (the actual section and its plain-language rendering). In a real
platform, each module/section-type (ICF eligibility, ICF risks, PLPS summary, SOC entry, ...)
would have its own instruction "family" like this — training the model to reliably produce that
section type's format and register, section after section, is exactly what instruction tuning
buys you over prompting a base model cold every time.

## 2. LoRA from scratch: `W + B·A`

A pretrained weight matrix `W` (shape `d_out x d_in`) is **frozen** — we never touch it. LoRA
represents the *update* needed to adapt the model as a product of two much smaller matrices:
`B` (`d_out x r`) and `A` (`r x d_in`), with rank `r << d_in, d_out`. The effective weight used
at inference is `W + B @ A`. Only `B` and `A` are ever trained.

We build a small (64x64) frozen "weight matrix" as a stand-in for one weight matrix inside a
much larger real transformer layer, and count parameters both ways.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

d_out, d_in = 64, 64
W = rng.normal(scale=0.02, size=(d_out, d_in))
print("Frozen base weight W shape:", W.shape, "params:", W.size)

def full_finetune_param_count(W):
    return W.size

r = 4
A = rng.normal(scale=0.02, size=(r, d_in))
B = np.zeros((d_out, r))  # LoRA convention: B initialized to zero so B@A starts at 0 (no change at init)

def lora_param_count(B, A):
    return B.size + A.size

full_params = full_finetune_param_count(W)
lora_params = lora_param_count(B, A)
print(f"Full fine-tune trainable params: {full_params}")
print(f"LoRA (rank={r}) trainable params: {lora_params}")
print(f"LoRA uses {lora_params/full_params:.1%} of the full fine-tune parameter count")

assert lora_params < full_params

In [ ]:
assert np.allclose(B @ A, 0)
print("OK: B initialized to zero -> B@A has no effect before training, matching LoRA init convention.")

x = rng.normal(size=(5, d_in))
out_frozen = x @ W.T
out_with_adapter = x @ (W + B @ A).T
assert np.allclose(out_frozen, out_with_adapter)
print("OK: at initialization, output with the LoRA adapter attached is identical to the frozen "
      "base model -- the adapter starts as a no-op and only diverges as training updates B, A.")

## 3. A toy training loop: can a low-rank adapter recover a low-rank update?

LoRA's premise is an empirical one: the update a model needs to adapt to a new, narrower task
(here, "always summarize protocol sections into strict PLPS-style plain language") tends to have
low **intrinsic rank** — you don't need a full-rank change to capture it. To demonstrate that
concretely, we construct a synthetic "ideal" target update that genuinely *is* low-rank (rank 4,
matching our adapter's rank), then run plain gradient descent on `B` and `A` only (never
touching `W`) and check how close the trained adapter gets.

In [ ]:
true_rank = 4
B_true = rng.normal(scale=0.05, size=(d_out, true_rank))
A_true = rng.normal(scale=0.05, size=(true_rank, d_in))
target_delta = B_true @ A_true  # the "ideal" weight update this fine-tune should learn, low-rank by construction
W_target = W + target_delta

lr = 0.05
for step in range(3000):
    W_eff = W + B @ A
    grad_W_eff = 2 * (W_eff - W_target)   # gradient of ||W_eff - W_target||^2 wrt W_eff
    grad_B = grad_W_eff @ A.T
    grad_A = B.T @ grad_W_eff
    B -= lr * grad_B
    A -= lr * grad_A

recon_error_lora = np.linalg.norm((W + B @ A) - W_target)
recon_error_frozen = np.linalg.norm(W - W_target)
print(f"Reconstruction error, frozen W only (no adaptation): {recon_error_frozen:.4f}")
print(f"Reconstruction error, after training the rank-{r} B@A adapter:  {recon_error_lora:.6f}")

assert recon_error_lora < recon_error_frozen * 0.1
print(f"\nOK: a rank-{r} adapter recovers a rank-{true_rank} target update almost exactly, "
      f"training only {lora_params/full_params:.1%} of the parameters a full dense update "
      f"would require.")

## 4. Guarded real-library usage

This notebook is intentionally numpy-only so it runs anywhere. A real fine-tuning run would use
Hugging Face `transformers` + `peft` against an actual LLaMA 3 checkpoint on a GPU. The cell
below checks whether those libraries happen to be available and, if not, explains what the real
call would look like rather than failing.

In [ ]:
try:
    import transformers  # noqa: F401
    import peft  # noqa: F401
    HAVE_REAL_LIBS = True
except ImportError:
    HAVE_REAL_LIBS = False

if HAVE_REAL_LIBS:
    print(
        "transformers/peft are available in this environment. A real LoRA fine-tuning run would "
        "look roughly like:\n\n"
        "  from peft import LoraConfig, get_peft_model\n"
        "  config = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj', 'v_proj'], "
        "lora_dropout=0.05)\n"
        "  model = get_peft_model(base_llama3_model, config)\n"
        "  # ... train `model` on the INSTRUCTION_DATASET-shaped examples from section 1 ...\n\n"
        "This notebook does not execute that path (no checkpoint download, no GPU) -- the numpy "
        "demo above is the mechanism it relies on."
    )
else:
    print(
        "transformers/peft are not installed in this environment, which is expected for a fully "
        "offline demo. The numpy demonstration above (Section 2-3) captures the actual mechanism: "
        "freeze the base weights, train a small low-rank B/A pair, and add B@A back in at inference "
        "time. A real run on LLaMA 3 would wrap `peft.LoraConfig(r=..., lora_alpha=..., "
        "target_modules=[...])` around the attention projection layers and train on data shaped "
        "exactly like INSTRUCTION_DATASET above, scaled up to thousands of examples per module/"
        "section-type."
    )

## 5. Tying it back

- The dataset in Section 1 is the *content* half of instruction tuning: real, human-quality
  `(instruction, input, output)` examples, one "family" per module/section-type (ICF, PLPS, SOC),
  built from historical documents plus synthetic augmentation for underrepresented cases.
- The LoRA demo in Sections 2-3 is the *mechanism* half: how you can specialize a large frozen
  model's behavior cheaply, training a small adapter instead of the whole network, and why that's
  the practical way to get several module-specific behaviors (an ICF adapter, a PLPS adapter, an
  SOC adapter) out of one shared LLaMA 3 base without paying full fine-tuning cost per module.
- The parameter-count gap you saw above (12.5% here, and far smaller in a real multi-billion-
  parameter model, since the ratio shrinks as `d` grows while `r` stays small) is the concrete
  number behind the claim "LoRA/QLoRA made fine-tuning LLaMA 3 for this domain practical."